In [0]:
class Bronze_lap_times():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze" 
    
    def __init__(self,folder_name,source):
        self.folder_name=folder_name
        self.source=source
    
    def get_schema(self):
        schema='''
            raceId INT NOT NULL,
            driverId INT NOT NULL,
            lap INT NOT NULL,
            position INT,
            time STRING,
            milliseconds INT
        '''
        return schema

    def read_data(self):
        df= (spark.readStream
             .format("csv")
             .schema(self.get_schema())
             .option("maxFilesPerTrigger", 1)
             #.option("rowsPerSecond", 1000)
             .option('header','true')
             .load(f"{self.main_path}/formula1_source/{self.folder_name}")
             )
        return df
        
    def process(self):
        print(f"\nStarting Bronze lap_times Stream...", end='')
        readDF = self.read_data()
        from pyspark.sql.functions import current_timestamp,lit
        readDF= (readDF.withColumn('LapTimesIngestionDate',current_timestamp())
                 .withColumn('source',lit(self.source))
                 )
        sQuery =  ( readDF.writeStream
                            .queryName("bronze-ingestion-lap")
                            .option("checkpointLocation", f"{self.main_path}/{self.bronze_path}/{self.folder_name}/checkpoint")
                            .outputMode("append")
                            #.option('path',f"{self.main_path}/{self.bronze_path}/{self.folder_name}")
                            .trigger(availableNow=True)
                            .toTable('formula1_race.bronze.lap_times')
                    ) 
        print("Done")
        return sQuery   

In [0]:
from pyspark.sql.streaming import StreamingQueryListener

class MyListener(StreamingQueryListener):
    def onQueryStarted(self, event):
        print(f"Query started: {event.id}")

    def onQueryProgress(self, event):
        print(f"Batch ID: {event.progress['batchId']}")
        print(f"Rows Read: {event.progress['numInputRows']}")
        print(f"Duration (ms): {event.progress['durationMs']}")
        print(f"Rows/sec: {event.progress['processedRowsPerSecond']}")

    def onQueryTerminated(self, event):
        print(f"Query terminated: {event.id}")

spark.streams.addListener(MyListener())

In [0]:
source='Ergast API'
Bronze_lap_times_instance = Bronze_lap_times("lap_times",source)
Squery_Bronze_lap_times =Bronze_lap_times_instance.process()
Squery_Bronze_lap_times.awaitTermination()
print("Successfully bronze-ingestion-circuits stream in running")
Squery_Bronze_lap_times.stop()